In [1]:
import pandas as pd

# =====================================================================
# 1. SETUP: Simulating incoming raw automated test lab data
# =====================================================================
source_file = "raw_diode_test_log.csv"
raw_data = """Diode_ID,Batch_Number,Voltage_V,Current_mA
D_01,B_74,1.20,8.5
D_02,B_74,1.45,4.2
D_03,B_75,1.15,12.1
D_04,B_75,2.10,3.1
D_05,B_76,1.30,9.0
D_06,B_76,1.85,15.4
"""
with open(source_file, "w", encoding="utf-8") as f:
    f.write(raw_data.strip())
print(f"[INFO] Source file '{source_file}' created.\n")

# =====================================================================
# 2. INGESTION & DIAGNOSTICS: Loading and inspecting data shape
# =====================================================================
df = pd.read_csv("raw_diode_test_log.csv", index_col='Diode_ID') 

print("--- Quick Visual Quality Check ---")
print(df.head(2))
print("\n--- Structural Telemetry Check ---")
df.info()
print("\n" + "="*50 + "\n")

# =====================================================================
# 3. ANALYSIS & FILTERING: Dynamic resistance math and failures sweep
# =====================================================================

# Compute dynamic resistance in Ohms (R = V / I * 1000)
df['Resistance_Ohms'] = (df['Voltage_V'] / df['Current_mA']) * 1000.0

# Isolate hardware components that violate the 250 Ohm safety limit
diode_failure_mask = df['Resistance_Ohms'] >= 250.0
failed_atp_diodes = df[diode_failure_mask]

# Sort the out-of-spec sub-table from highest resistance to lowest
failed_atp_diodes = failed_atp_diodes.sort_values(by='Resistance_Ohms', ascending=False)

# =====================================================================
# 4. EXPORT & REPORTING: Cleaning and writing final report to disk
# =====================================================================

# Isolate only the relevant parameters for the development team layout
final_failed_report = failed_atp_diodes.loc[:, ['Batch_Number', 'Resistance_Ohms']]

# Write the final sorted anomalies log straight to your computer drive
output_filename = "diode_failures_report.csv"
final_failed_report.to_csv(output_filename, index=True)

print("=== HARDWARE COMPLIANCE SCAN RESULTS ===")
print(f"Total Out-of-Spec Diodes Found: {len(final_failed_report)}")
print("--------------------------------------------------\n")
print(final_failed_report)
print("\n--------------------------------------------------")
print(f"[SUCCESS] Anomaly report generated and saved as '{output_filename}'")
print("==================================================")


[INFO] Source file 'raw_diode_test_log.csv' created.

--- Quick Visual Quality Check ---
         Batch_Number  Voltage_V  Current_mA
Diode_ID                                    
D_01             B_74       1.20         8.5
D_02             B_74       1.45         4.2

--- Structural Telemetry Check ---
<class 'pandas.DataFrame'>
Index: 6 entries, D_01 to D_06
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Batch_Number  6 non-null      str    
 1   Voltage_V     6 non-null      float64
 2   Current_mA    6 non-null      float64
dtypes: float64(2), str(1)
memory usage: 240.0 bytes


=== HARDWARE COMPLIANCE SCAN RESULTS ===
Total Out-of-Spec Diodes Found: 2
--------------------------------------------------

         Batch_Number  Resistance_Ohms
Diode_ID                              
D_04             B_75       677.419355
D_02             B_74       345.238095

--------------------------------------------------
